<h1 style="text-align: center; font-family: Arial; font-weight: bold; color:green; font-size:36px;">Renewable Energy Forecasting
    Dashboard for the Republic of Ireland</h1>

<b>Author:</b> J.A Montuya<br />
<b>Student ID:</b> 2025040<br />
<h3>Abstract:</h3>

In [2]:
# Import Libraries
import dash
from dash import dcc, html, Dash
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

import webbrowser

In [3]:
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Load Datasets

In [5]:
# Import Datasets
eu_prices_df = pd.read_csv('./Datasets/electricity-prices-by-usertype-eu.csv')

## Data Preparation

### 1. Electricity Prices in EU in Euros per KWH

In [8]:
# Country Code Mapping
country_code_dict = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CY": "Cyprus",
    "CZ": "Czech_Republic",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "EL": "Greece",
    "ES": "Spain",
    "EU27_2020": "European_Union",
    "FI": "Finland",
    "FR": "France",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia"
}

# Select usable columns
eu_prices_df=eu_prices_df[['TIME_PERIOD','geo', 'unit', 'OBS_VALUE','currency']]

# Add feature countries by mapping country code to country name 
eu_prices_df['country'] = eu_prices_df['geo'].map(country_code_dict)

# Remove empty rows
eu_prices_df.dropna(axis=0, inplace=True)

# Drop Duplicate values
eu_prices_df = eu_prices_df.drop_duplicates(subset=['country', 'TIME_PERIOD'])

# Restructure dataframe to make years as columns and countries to records
eu_prices_pivot_df = eu_prices_df[['country','TIME_PERIOD','OBS_VALUE']].pivot(index='country', columns='TIME_PERIOD', values='OBS_VALUE')

# Sort Values
eu_prices_pivot_df = eu_prices_pivot_df.sort_index()

# Show 5 records
eu_prices_pivot_df.head()

TIME_PERIOD,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
country,,,,,,,,,,,,
Austria,0.2082,0.2021,0.2009,0.2034,0.1950,0.1966,0.2034,0.2111,0.2216,0.2249,0.2653,0.2731
Belgium,0.2173,0.2097,0.2126,0.2544,0.2857,0.2824,0.2839,0.2792,0.2702,0.3437,0.4350,0.3354
Bulgaria,0.0924,0.0832,0.0942,0.0956,0.0955,0.0979,0.0997,0.0997,0.1024,0.1093,0.1138,0.1187
Croatia,0.1372,0.1312,0.1317,0.1311,0.1196,0.1311,0.1321,0.1301,0.1291,0.1354,0.1480,0.1472
Cyprus,0.2760,0.2291,0.1957,0.1527,0.1863,0.1893,0.2203,0.2133,0.1976,0.2607,0.3739,0.3241


## Data Visualisation

### 1. Electricity Prices in EU in Euros per KWH

In [79]:
def check_eu_prices():
    # Convert TIME_PERIOD as string
    eu_prices_df['TIME_PERIOD'] = eu_prices_df['TIME_PERIOD'].astype(str)

    # Sort by year and value
    eu_prices_df_sorted = eu_prices_df.sort_values(by=['TIME_PERIOD', 'OBS_VALUE'], ascending=[True, True])
    
    # Create bar plot
    fig = px.bar(
        data_frame=eu_prices_df_sorted,
        x='OBS_VALUE',
        y='country',
        orientation='h', 
        color='OBS_VALUE',  
        color_continuous_scale='Magma_r',
        animation_frame='TIME_PERIOD',  # Animate by year
        range_x=[eu_prices_df['OBS_VALUE'].min(), eu_prices_df['OBS_VALUE'].max()],
        width=600,   
        height=800,
          labels={
            'OBS_VALUE': 'Electricity Price (Euros per kWh)', 
            'TIME_PERIOD': 'Year',
            'country': 'Country'
        }
    )

    fig.update_layout(
        title=f'Electricity Prices by Country (EU)',
        xaxis_title='Price per KWH (Euro)',
        yaxis_title='Country',
        showlegend=True,
        template='plotly_white',
        paper_bgcolor='rgba(0,0,0,0)',  # Transparent outside the plot
        plot_bgcolor='rgba(0,0,0,0)'   
    )
    
    return fig

## Dashboard Implementation using Plotly Dash

In [81]:
# Build Dash App
app = Dash(__name__)

app.layout = html.Div(style={
    'backgroundColor': '#f0f0f5',
    'minHeight': '100vh',
    'padding': '0',
    'fontFamily': 'Arial, sans-serif'
}, children=[

    # Header
    html.Div("Renewable Energy Forecast for the Republic of Ireland", style={
        'backgroundColor': 'rgb(80, 200, 120)',
        'color': 'white',
        'padding': '20px',
        'fontSize': '28px',
        'fontWeight': 'bold',
        'textAlign': 'center',
        'borderRadius': '10px',
        'boxShadow': '0 4px 8px rgba(0, 0, 0, 0.2)'
    }),
     html.Div([
        # Container div holding both graphs in a row
        html.Div([
            # Left graph
            html.Div([
                dcc.Graph(figure=check_eu_prices(), config={
                    'scrollZoom': False,
                    'displayModeBar': False
                }, style={'height': '800px'})
            ], style={
                'width': '32%',
                'display': 'inline-block',
                'padding': '10px',
                'backgroundColor': 'white',
                'borderRadius': '10px',
                'boxShadow': '0 4px 8px rgba(0,0,0,0.1)',
                'verticalAlign': 'top'
            }),
    
            # Right graph
            html.Div([
                dcc.Graph(figure=check_eu_prices(), config={
                    'scrollZoom': False,
                    'displayModeBar': False
                }, style={'height': '800px'})
            ], style={
                'width': '62%',
                'display': 'inline-block',
                'padding': '10px',
                'backgroundColor': 'white',
                'borderRadius': '10px',
                'boxShadow': '0 4px 8px rgba(0,0,0,0.1)',
                'verticalAlign': 'top'
            })
        ], style={
            'width': '100%',
            'display': 'flex',
            'justifyContent': 'space-around'
        })
    ])
])

if __name__ == '__main__':
    app.run(debug=True)
    #webbrowser.open("http://127.0.0.1:8050")